<a href="https://colab.research.google.com/github/aqibkhan777/-Sign-Language-Detection-Model-with-Machine-Learning/blob/main/sign_language_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**[Teaching Machines to Understand Sign Language](https://www.sciencebuddies.org/)**

This notebook was developed by Science Buddies [www.sciencebuddies.org](https://www.sciencebuddies.org/) as part of a science project to allow students to explore and learn about artificial intelligence. For personal use, this notebook can be downloaded and modified with attribution. For all other uses, please see our [Terms and Conditions of Fair Use](https://www.sciencebuddies.org/about/terms-and-conditions-of-fair-use).  

**Troubleshooting tips**
*   Read the written instructions at Science Buddies and the text and comments on this page carefully.
*   If you make changes that break the code, you can download a fresh copy of this notebook and start over.

*   If you are using this notebook for a science project and need help, visit our [Ask an Expert](https://www.sciencebuddies.org/science-fair-projects/ask-an-expert-intro) forum for assistance.

## **How To Use This Notebook**

This notebook contains text fields, like this one, that give you information about the project and instructions.

In [ ]:
# There are also code blocks, like this one.

# The green text in a code block are comments. Comments are descriptions of what the code does.

# The non-green text in a code block is the Python code. Click on the triangle in the top left corner to run this code block.

print("Congratulations, you ran a code block! Try changing the text in the code and running it again.")

Congratulations, you ran a code block! Try changing the text in the code and running it again.


# Installing Dependencies

In [ ]:
!pip uninstall -y mediapipe
!pip install -U "numpy<2" "protobuf<5"
!pip install --no-cache-dir "mediapipe==0.10.21"

Found existing installation: mediapipe 0.10.21
Uninstalling mediapipe-0.10.21:
  Successfully uninstalled mediapipe-0.10.21
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 189.4 MB/s eta 0:00:00


In [ ]:
# This command installs the TensorFlow library using pip (Python's package installer)
!pip install tensorflow

In [ ]:
# Install the OpenCV library, which is used for computer vision tasks such as image and video processing.
!pip install opencv-python

# 1. Importing Libraries

In [ ]:
# General libraries
import os  # For interacting with the operating system (e.g., file handling)
import time  # For time-related functions
import numpy as np  # For numerical operations and handling arrays
import scipy.stats  # For statistical functions (e.g., mode, z-test)
import random
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt  # For plotting graphs and images
from IPython.display import display, Image  # For displaying images and other media in Jupyter/Colab

# Computer Vision Libraries
import cv2  # For computer vision tasks such as image and video processing
import mediapipe as mp  # For using MediaPipe's machine learning models for pose detection and more
from google.colab.patches import cv2_imshow  # For displaying images with OpenCV in Google Colab

# Machine Learning Libraries
from sklearn.model_selection import train_test_split  # For splitting data into training and testing sets
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score  # For model evaluation

# Deep Learning Libraries (TensorFlow and Keras)
from tensorflow.keras.utils import to_categorical  # For converting labels to categorical format (one-hot encoding)
from tensorflow.keras.models import Sequential  # For defining the neural network architecture
from tensorflow.keras.layers import LSTM, Dense  # For adding LSTM and Dense layers to the model
from tensorflow.keras.callbacks import TensorBoard  # For logging training progress for TensorBoard
from tensorflow.keras.models import load_model  # For loading pre-trained models

In [ ]:
# Import the 'drive' module from Google Colab to interact with Google Drive
from google.colab import drive
import os # Import the os module to interact with the operating system

# Check if the mount point directory exists and remove it if it does,
# to ensure a clean state before attempting to mount Google Drive.
if os.path.exists('/content/drive'):
    print("Removing existing /content/drive directory to ensure a clean mount.")
    !rm -rf /content/drive

# Mount Google Drive to access files stored in it. This will create a folder 'drive' in the '/content' directory.
# After running this, you will be prompted to authorize Colab to access your Google Drive.
drive.mount('/content/drive', force_remount=True)

Removing existing /content/drive directory to ensure a clean mount.
rm: cannot remove '/content/drive/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.Trash-0': Directory not empty
rm: cannot remove '/content/drive/.Encrypted/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/.Encrypted/MyDrive': Operation canceled
Mounted at /content/drive


In [ ]:
# Set the base directory to the specified path on Google Drive.
# This is the location where the 'sign_language_detection' folder is stored within the 'MyDrive' folder.
base_dir = "/content/drive/MyDrive/sign_language_detection"

test_videos_path = os.path.join(base_dir, 'test_videos')
if not os.path.exists(test_videos_path):
    os.makedirs(test_videos_path)

DATA_PATH = os.path.join(base_dir, 'MP_Data')

# 2. Detecting Keypoints using MP Holistic

Code Block 2A

In [ ]:
# Initialize the Holistic model from MediaPipe for full-body pose detection.
# This model can detect landmarks for the face, hands, and body.
import mediapipe as mp # Added this import statement
mp_holistic = mp.solutions.holistic

# Initialize the drawing utilities from MediaPipe to draw the landmarks on images or videos.
# These utilities help visualize the pose detection results by drawing the keypoints and connections.
mp_drawing = mp.solutions.drawing_utils

# This function processes an image using a MediaPipe model to detect landmarks/poses and returns the processed image and the results.
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

# This function draws the detected pose, left hand, and right hand landmarks with their connections on the given image.
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

# This function draws the pose, left hand, and right hand landmarks with custom styling (color, thickness, and circle radius)
# on the given image, using the MediaPipe Holistic model's results.
def draw_styled_landmarks(image, results):
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             )
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             )
    # Draw right hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             )

Code Block 2B

In [ ]:
# TODO: Change the name of the test video to ensure the code is running
import os
video_name = "IMG_6750.MOV"
video_path = os.path.join(test_videos_path, video_name)

# Try to open the video
cap = cv2.VideoCapture(video_path)

# Check if the video opened successfully
if not cap.isOpened():
    raise FileNotFoundError(f"Error: Cannot open video file '{video_path}'. Please check if the file exists and the path is correct.")

# Set MediaPipe model
mp_holistic = mp.solutions.holistic
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        frame = cv2.flip(frame, 0)

        if not ret:
            break

        # ✅ Flip ONCE at the very beginning (fixes input video orientation)
        frame = cv2.flip(frame, 0)

        # Make detections
        image, results = mediapipe_detection(frame, holistic)

        # Draw landmarks on the frame
        draw_styled_landmarks(image, results)

        # Convert frame to RGB (to display in Colab)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Display the frame
        plt.figure(figsize=(10, 10))
        plt.imshow(image_rgb)
        plt.axis('off')
        display(plt.gcf())
        plt.close()

        # Pause for a short moment to simulate frame rate
        cv2.waitKey(1)

    cap.release()

# 3. Extracting Keypoint Values

Code Block 3A

In [ ]:
# Initialize an empty list to store pose landmark data
pose = []

# Loop through each landmark in the detected pose landmarks
for res in results.pose_landmarks.landmark:
    # Create a numpy array with the x, y, z coordinates and visibility of each landmark
    test = np.array([res.x, res.y, res.z, res.visibility])

    # Append the numpy array for the current landmark to the 'pose' list
    pose.append(test)

# Create a numpy array for pose landmarks, flattening the x, y, z coordinates and visibility for each landmark.
# If pose landmarks are not detected, return a zero array of length 132 (assuming 33 landmarks, each with 4 values: x, y, z, visibility).
pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)

# Create a numpy array for face landmarks, flattening the x, y, z coordinates for each landmark.
# If face landmarks are not detected, return a zero array of length 1404 (assuming 468 landmarks, each with 3 values: x, y, z).
face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404)

# Create a numpy array for left hand landmarks, flattening the x, y, z coordinates for each landmark.
# If left hand landmarks are not detected, return a zero array of length 63 (21 landmarks, each with 3 values: x, y, z).
lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)

# Create a numpy array for right hand landmarks, flattening the x, y, z coordinates for each landmark.
# If right hand landmarks are not detected, return a zero array of length 63 (21 landmarks, each with 3 values: x, y, z).
rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)

# This function extracts keypoints (pose, face, left hand, right hand) from the results of MediaPipe landmarks,
# flattens the data into 1D arrays, and concatenates them into a single array for further processing.
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

# 4. Setting up Folders for Collection

Code Block 4A

In [ ]:
# TODO: Define the set of actions (sign language gestures) to detect
actions = np.array(['hello', 'thank_you'])

# Path for exported data (numpy arrays will be saved here)
DATA_PATH = os.path.join(base_dir, 'MP_Data')
VIDEO_PATH = os.path.join(base_dir, 'videos')

# Create the data directory if it doesn't exist
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

# Number of sequences (videos) to record per action
no_sequences = 30

# Number of frames per video sequence
sequence_length = 30

# Starting index for naming folders that store each sequence
start_folder = 30

# Create folder structure for each action and each sequence
for action in actions:
    # Path for this action's data
    action_path = os.path.join(DATA_PATH, action)
    video_path = os.path.join(VIDEO_PATH, action)

    # Create action folder if it doesn't exist
    if not os.path.exists(action_path):
        os.makedirs(action_path)

    # Create video folder if it doesn't exist
    if not os.path.exists(video_path):
        os.makedirs(video_path)

    # Find existing numbered folders (sequences) for this action
    existing_dirs = [d for d in os.listdir(action_path) if d.isdigit()]

    # Determine the current max sequence number to avoid overwriting
    if existing_dirs:
        dirmax = np.max(np.array(existing_dirs).astype(int))
    else:
        dirmax = 0

    # Create subfolders for each new sequence
    for sequence in range(0, no_sequences):
        seq_path = os.path.join(action_path, str(dirmax + sequence))
        if not os.path.exists(seq_path):
            os.makedirs(seq_path)

# 5. Collecting Keypoint Values for Training and Testing

Code Block 5A

In [ ]:
# Paths for original videos (where the raw sign language videos will be stored)
base_video_path = os.path.join(base_dir, 'videos')

# Create the base 'videos' directory if it doesn't exist
if not os.path.exists(base_video_path):
    os.makedirs(base_video_path)

# Loop through each action (e.g., 'hello', 'thankyou', 'seeyoulater')
for action in actions:
    # Define paths for the videos and the extracted data (keypoints)
    action_video_path = os.path.join(base_video_path, action)
    action_data_path = os.path.join(DATA_PATH, action)
    os.makedirs(action_data_path, exist_ok=True)  # Ensure the data path exists

    # Create 30 subfolders (0 to 29) under the action's data directory, one for each frame position
    for i in range(no_sequences):
        sequence_folder = os.path.join(action_data_path, str(i))
        os.makedirs(sequence_folder, exist_ok=True)

    video_counter = 0  # Keeps track of how many videos have been processed

    # Loop through all video files in the action's video folder
    for video_file in os.listdir(action_video_path):
        print(f"\n▶ Processing {video_file}")

        full_video_path = os.path.join(action_video_path, video_file)

        # Load the video using OpenCV
        cap = cv2.VideoCapture(full_video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Get total number of frames
        print(f"Total frames: {total_frames}")

        # Generate 30 evenly spaced frame indices from the video
        target_frame_count = 30
        frame_indices = np.linspace(0, total_frames - 1, target_frame_count, dtype=int)

        # Set up MediaPipe Holistic model for keypoint detection
        with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
            for i, frame_index in enumerate(frame_indices):
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)  # Jump to the desired frame
                ret, frame = cap.read()
                if not ret:
                    print(f"⚠ Could not read frame {frame_index}")
                    continue  # Skip if frame can't be read

                # Run detection and draw landmarks
                image, results = mediapipe_detection(frame, holistic)
                draw_styled_landmarks(image, results)

                # Extract keypoints from results
                keypoints = extract_keypoints(results)

                # Save the keypoints to a .npy file in the appropriate subfolder
                npy_path = os.path.join(action_data_path, str(i), f"{video_counter}.npy")
                np.save(npy_path, keypoints)
                print(f"✅ Saved to {npy_path}")

        # Release the video capture object
        cap.release()
        video_counter += 1  # Move to next video


▶ Processing IMG_6756.MOV
Total frames: 120
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/0/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/1/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/2/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/3/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/4/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/5/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/6/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/7/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/8/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/9/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/10/0.npy
✅ Saved to /content/drive/MyDrive/sign_language_detection/MP_Data/hello/11

# 6. Preprocessing Data and Creating Labels and Features

Code Block 6A

In [ ]:
# Create a dictionary that maps each action label (e.g., 'hello') to a unique integer
# This is useful for converting string labels into numeric format for model training
label_map = {label: num for num, label in enumerate(actions)}

# Display the resulting label map
label_map

{np.str_('hello'): 0, np.str_('thank_you'): 1}

Code Block 6B

In [ ]:
# Initialize lists to store sequences of keypoints and their corresponding labels
sequences, labels = [], []

# Loop through each action (e.g., 'hello', 'thank_you', 'see_you_later')
for action in actions:
    # Get the list of sequence folders (0, 1, ..., 29) for this action
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        window = []  # Temporary list to hold the 30 frames of one video sequence

        # Loop through each frame in the sequence (expected to be 30 frames)
        for frame_num in range(sequence_length):
            # Load the keypoints for this frame from its .npy file
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)  # Append the keypoints to the current sequence

        # After collecting all 30 frames, add the full sequence to the dataset
        sequences.append(window)

        # Add the corresponding label (numeric) for this action
        labels.append(label_map[action])

# Convert the list of sequences into a NumPy array for model training
# Each sequence is a 30-frame window, where each frame contains the extracted keypoints
X = np.array(sequences)

# Convert the list of numeric labels into one-hot encoded format
# For example, label 0 becomes [1, 0, 0], label 1 becomes [0, 1, 0], etc.
y = to_categorical(labels).astype(int)

Code Block 6C

In [ ]:
# Split the dataset into training and testing sets
# 95% of the data will be used for training, 5% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

# Print the shapes of the resulting datasets to verify the split
print("X_train shape:", X_train.shape)  # Shape: (num_train_samples, num_frames, num_features)
print("X_test shape:", X_test.shape)    # Shape: (num_test_samples, num_frames, num_features)
print("y_train shape:", y_train.shape)  # Shape: (num_train_samples, num_classes)
print("y_test shape:", y_test.shape)    # Shape: (num_test_samples, num_classes)

X_train shape: (57, 30, 1662)
X_test shape: (3, 30, 1662)
y_train shape: (57, 2)
y_test shape: (3, 2)


# 7. Building and Training an LSTM Neural Network

Code Block 7A

In [ ]:
# Set the directory where TensorBoard logs will be saved
log_dir = os.path.join('Logs')

# Create a TensorBoard callback to log training metrics (loss, accuracy, etc.)
# This allows you to visualize training progress in TensorBoard
tb_callback = TensorBoard(log_dir=log_dir)

# Build a sequential neural network model
model = Sequential()

# First LSTM layer with 64 units; returns sequences to feed into next LSTM
# Input shape is (30 time steps, 1662 features per frame)
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30,1662)))

# Second LSTM layer with 128 units; also returns sequences
model.add(LSTM(128, return_sequences=True, activation='relu'))

# Third LSTM layer with 64 units; does not return sequences (only final output is passed on)
model.add(LSTM(64, return_sequences=False, activation='relu'))

# Fully connected dense layer with 64 units and ReLU activation
model.add(Dense(64, activation='relu'))

# Fully connected dense layer with 32 units and ReLU activation
model.add(Dense(32, activation='relu'))

# Output layer with number of units equal to number of actions (e.g., 3),
# and softmax activation for multi-class classification
model.add(Dense(actions.shape[0], activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Code Block 7B

In [ ]:
# Compile the model with the following settings:
# 'Adam' optimizer is used, which is an adaptive learning rate method commonly used in training deep learning models
model.compile(optimizer='Adam',
              # Use 'categorical_crossentropy' as the loss function, suitable for multi-class classification problems
              loss='categorical_crossentropy',
              # Track 'categorical_accuracy' during training to evaluate how well the model is classifying the correct action
              metrics=['categorical_accuracy'])

In [ ]:
# Train the model using the training data
# X_train: Input data (sequences of keypoints)
# y_train: Target labels (one-hot encoded labels for the actions)
# epochs=200: The model will train for 200 epochs (iterations over the entire dataset)
# callbacks=[tb_callback]: This will use the TensorBoard callback to log training progress and metrics for visualization
model.fit(X_train, y_train, epochs=200, callbacks=[tb_callback])

Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 209ms/step - categorical_accuracy: 0.4561 - loss: 0.7041
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 425ms/step - categorical_accuracy: 0.4561 - loss: 0.7289
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 324ms/step - categorical_accuracy: 0.4912 - loss: 1.5331
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 294ms/step - categorical_accuracy: 0.5263 - loss: 0.8734
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 273ms/step - categorical_accuracy: 0.4386 - loss: 0.6957
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 364ms/step - categorical_accuracy: 0.6667 - loss: 1.6403
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 274ms/step - categorical_accuracy: 0.4912 - loss: 3.8407
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 797ms/step - categorical_accuracy: 0.4211 - loss: 3.2871
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 460ms/step - categorical_accuracy: 0.4386 - loss: 1.6230
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 332ms/step - categorical_accuracy: 0.3509 - loss: 7.5550
Epoch 11/200
2/2 ━━━━━━━━━━━━

# 8. Making Predictions

Code Block 8A

In [ ]:
# Use the trained model to make predictions on the test data
# X_test: Input data (test set of sequences of keypoints)
# The model predicts the output (probabilities for each class) for each sample in the test set
y_pred = model.predict(X_test)

# Display the predictions (e.g., probabilities for each class for each test sample)
y_pred.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 510ms/step


(3, 2)

Code Block 8B

In [ ]:
# Get the predicted action label for the 1st sample in the test set (index 0)
# np.argmax(res[0]) returns the index of the highest predicted probability (most likely class) for the 1st sample
# The index is used to retrieve the corresponding action label from the 'actions' array
actions[np.argmax(y_pred[0])]  # Changed index to 0 to access a valid prediction

np.str_('hello')

Code Block 8C

In [ ]:
# Get the true action label for the 1st sample in the test set (index 0)
# np.argmax(y_test[0]) returns the index of the highest value in the one-hot encoded true label for the 1st sample
# The index is used to retrieve the corresponding action label from the 'actions' array
actions[np.argmax(y_test[0])]

np.str_('hello')

# 9. Saving Weights

Code Block 9A

In [ ]:
# Define the file path where the trained model will be saved
# The model will be saved as a .h5 file (HDF5 format), which stores the model architecture, weights, and optimizer state
model_path = os.path.join(base_dir, 'my_model.keras')

# Save the trained model to the specified path
# This allows you to later load the model for inference or continue training
model.save(model_path)

Code Block 9B

In [ ]:
# Load the saved model from the specified file path
model_path = os.path.join(base_dir, 'my_model.keras')
# 'model_path' points to the location where the model was previously saved (e.g., 'action.h5')
# The model is loaded along with its architecture, weights, and optimizer state, making it ready for inference or further training
model = load_model(model_path)

# 10. Evaluating Using a Confusion Matrix and Accuracy

Code Block 10A

In [ ]:
# Use the trained model to make predictions on the test data (X_test)
# The model outputs the predicted probabilities for each class in the test set
# y_pred will contain the predicted probabilities or class labels for each sample in X_test
y_pred = model.predict(X_test)

# Convert the one-hot encoded true labels (y_test) into class labels (indices)
# np.argmax(y_test, axis=1) finds the index of the maximum value along the second axis (for each sample)
# This index corresponds to the true class label (e.g., [1, 0, 0] -> class 0, [0, 1, 0] -> class 1)
y_true = np.argmax(y_test, axis=1).tolist()

# Convert the predicted probabilities (y_pred) into predicted class labels (indices)
# np.argmax(y_pred, axis=1) finds the index of the highest predicted probability for each sample
# This index corresponds to the predicted class label
y_pred = np.argmax(y_pred, axis=1).tolist()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step


Code Block 10B

In [ ]:
# Calculate multilabel confusion matrix
conf_matrix = multilabel_confusion_matrix(y_true, y_pred)

for idx, matrix in enumerate(conf_matrix):
    print(f"\nConfusion Matrix for class '{actions[idx]}':")
    df = pd.DataFrame(matrix,
                      index=["Actual Negative", "Actual Positive"],
                      columns=["Predicted Negative", "Predicted Positive"])
    print(df)


Confusion Matrix for class 'hello':
                 Predicted Negative  Predicted Positive
Actual Negative                   1                   0
Actual Positive                   0                   2

Confusion Matrix for class 'thank_you':
                 Predicted Negative  Predicted Positive
Actual Negative                   2                   0
Actual Positive                   0                   1


Code Block 10C

In [ ]:
# Calculate the accuracy score to evaluate the model's performance
# The accuracy score measures the ratio of correctly predicted labels to the total number of samples
# It compares the true labels (y_true) with the predicted labels (y_pred) and returns the percentage of correct predictions
accuracy_score(y_true, y_pred)

1.0

# 11. Testing with Videos

Code Block 11A

In [ ]:
# Paths
test_video = "IMG_6750.MOV" # TODO: Change to the name of your video file
base_test_path = "/content/drive/MyDrive/sign_language_detection/test_videos"
input_video_path = os.path.join(base_test_path, test_video)
output_video_path = os.path.join(base_test_path, "output_video.mp4")
os.makedirs(base_test_path, exist_ok=True)

colors = [tuple(random.randint(0, 255) for _ in range(3)) for _ in actions]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()

    for num, prob in enumerate(res):
        color = colors[num]
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), color, -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

    return output_frame


# Detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

# Load video
cap = cv2.VideoCapture(input_video_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Output video setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

# Mediapipe setup
mp_holistic = mp.solutions.holistic
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # ✅ Flip ONCE at the very beginning (fixes input video orientation)
        frame = cv2.flip(frame, 0)

        # Detection
        image, results = mediapipe_detection(frame, holistic)
        draw_styled_landmarks(image, results)

        # Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            predictions.append(np.argmax(res))

            if np.unique(predictions[-10:])[0] == np.argmax(res):
                if res[np.argmax(res)] > threshold:
                    if len(sentence) == 0 or actions[np.argmax(res)] != sentence[-1]:
                        sentence.append(actions[np.argmax(res)])

            if len(sentence) > 5:
                sentence = sentence[-5:]

            image = prob_viz(res, actions, image, colors)

        # Add label
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        # ✅ Save the frame (already flipped correctly)
        out.write(image)

    cap.release()
    out.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━